# App
> FastHTML shell: dock (icons **1** and **4** implemented in this package), expanded visualization area, and chat that drives Lisette plus optional Jupyter notebook cell appends. Vault/graph routes from the standalone ``string therapy1`` app are omitted here until a ``db`` layer exists in ``string_therapy``.

In [ ]:
#| default_exp app

## Imports and environment

Load ``.env`` via ``therapy_config`` before importing icons that read env vars. Icon **1** is the markdown notebook UI; icon **4** embeds Jupyter Lab and exposes helpers used by the chat handler.

In [ ]:
#| export
from __future__ import annotations

import inspect
import os
import threading
import webbrowser

from fasthtml.common import *
from monsterui.all import *
from starlette.responses import PlainTextResponse, Response

import string_therapy.therapy_config as tc

tc.load_environment()

from string_therapy.icon1 import (
    register_routes as register_icon1_routes,
    viz as icon1_viz,
)
from string_therapy.icon4 import (
    ICON_ID as ICON4_ICON_ID,
    SESSION_NOTEBOOK_KEY,
    append_chat_turn_to_notebook,
    chat as icon4_chat,
    default_notebook_path,
    get_oob_html,
    resolve_notebook_path,
    register_routes as register_icon4_routes,
    viz as icon4_viz,
)

## App factory, registry, dock

``fast_app`` supplies ``rt`` for route decorators. We register each icon’s HTTP partials, then map slot numbers to **callables** (``viz``), not dataclasses. ``_dock_icon_ids`` filters ``ST_ICON_IDS`` / defaults so only implemented slots get buttons—currently **1** and **4**.

In [ ]:
#| export
hdrs = list(Theme.blue.headers())
hdrs.append(Script(src="https://d3js.org/d3.v7.min.js"))

app, rt = fast_app(hdrs=hdrs, canonical=False)

register_icon1_routes(app)
register_icon4_routes(app)

@app.get("/favicon.ico")
def favicon():
    return Response(status_code=204)


@app.get("/health")
def health():
    return PlainTextResponse("ok")


_VIZ = {
    1: icon1_viz,
    4: icon4_viz,
}


def _dock_icon_ids() -> tuple[int, ...]:
    """Dock buttons: ``ST_ICON_IDS`` filtered to icons implemented in ``_VIZ``, else ``(1, 4)``."""
    want = tc.dock_icon_ids()
    implemented = tuple(i for i in want if i in _VIZ)
    return implemented if implemented else (1, 4)


def _ensure_sid(session: dict, key: str) -> str:
    if key not in session:
        session[key] = os.urandom(8).hex()
    return session[key]


def render_icon(icon_id: int, session: dict):
    v = _VIZ.get(int(icon_id))
    if not v:
        return Div("Not Found")
    session["active_icon"] = int(icon_id)
    sk = f"icon{int(icon_id)}_sid"
    sid = _ensure_sid(session, sk)
    params = inspect.signature(v).parameters
    if "session" in params:
        return v(sid, session=session)
    return v(sid)


def _app_icon_button(icon_id: int) -> Button:
    """Tiny dock buttons — minimal chrome so the stack fits one viewport."""
    return Button(
        P(f"#{icon_id}", cls="text-[8px] font-bold leading-none"),
        type="button",
        hx_get=f"/expand/{icon_id}",
        hx_target="#expanded-space",
        hx_swap="innerHTML",
        cls=(
            "flex h-7 w-7 shrink-0 items-center justify-center rounded-md p-0 "
            "btn btn-xs btn-outline btn-primary min-h-0 min-w-0"
        ),
    )


def _app_icon_strip() -> Div:
    """Dock between the expanded viz (Lab / notebook) and the chat form."""
    return Div(
        cls="flex shrink-0 justify-center py-0.5",
        id="app-icon-dock",
    )(
        Div(
            cls="flex items-center gap-1 rounded-full border border-base-300/80 bg-base-200/80 px-1.5 py-0.5 shadow-sm",
        )(*[_app_icon_button(i) for i in _dock_icon_ids()]),
    )

## Chat and Lab shell helpers

``chat_icon4`` runs Lisette and builds an optional OOB fragment for ``#icon4-viz``. The expanded area mounts **both** icon panels (Lab iframe uses ``hx-preserve``); ``GET /expand/{id}`` swaps ``#expanded-space`` to show one panel and hide the other. ``_expanded_space_icon4_oob`` restores the Lab view after notebook writes when needed.
## Chat and Lab shell helpers

``chat_icon4`` runs Lisette and builds an optional OOB fragment for ``#icon4-viz``. The expanded area mounts **both** icon panels (Lab iframe uses ``hx-preserve``); ``GET /expand/{id}`` swaps ``#expanded-space`` to show one panel and hide the other. ``_expanded_space_icon4_oob`` restores the Lab view after notebook writes when needed.


In [ ]:
#| export
def chat_icon4(session: dict, msg: str, max_tokens: int):
    """Lab LLM + OOB fragment for ``#icon4-viz`` (same pattern as ``string therapy1``)."""
    sid = _ensure_sid(session, "icon4_sid")
    try:
        text = str(icon4_chat(sid, msg, max_tokens=max_tokens))
    except Exception as e:
        text = f"Error: {e}"
    script, div = get_oob_html(session, sid)
    oob = Div(NotStr(script), NotStr(div), id="icon4-viz", hx_swap_oob="true")
    return text, oob


def _chat_notebook_default(session) -> str:
    return resolve_notebook_path(None, session) or default_notebook_path()


def _expanded_space(session, active_icon_id: int) -> Div:
    """Expanded-space content.

    Keep the Jupyter iframe alive across dock switches by always including icon4's iframe
    markup (the iframe itself is marked ``hx-preserve``), and show/hide panels via CSS.
    """
    sid4 = _ensure_sid(session, "icon4_sid")
    icon4_node = icon4_viz(sid4, session=session)

    icon1_node = None
    if int(active_icon_id) == 1:
        sid1 = _ensure_sid(session, "icon1_sid")
        icon1_node = icon1_viz(sid1, session=session)

    panel_cls = "p-0 w-full flex-1 min-h-0 overflow-hidden flex flex-col"
    return Div(
        Input(
            name="active_icon",
            value=str(int(active_icon_id)),
            type="hidden",
            id="active-icon-input",
        ),
        Div(
            Div(
                icon1_node or "",
                cls="w-full flex-1 min-h-0 flex flex-col",
                id="icon1-viz",
            ),
            id="icon1-panel",
            cls=f"{panel_cls}{'' if int(active_icon_id) == 1 else ' hidden'}",
        ),
        Div(
            icon4_node,
            id="icon4-panel",
            cls=f"{panel_cls}{'' if int(active_icon_id) == ICON4_ICON_ID else ' hidden'}",
        ),
        cls="w-full flex-1 min-h-0 flex flex-col",
    )


def _chat_status_inner_html(text: str, *, ok: bool | None):
    """Body swapped into ``#chat-jupyter-status``."""
    if ok is True:
        c = "text-[#00ff00]"
    elif ok is False:
        c = "text-red-400"
    else:
        c = "text-white/50"
    return Span(text or "\u00a0", cls=c)


def _expanded_space_icon4_oob(session) -> Div:
    """HTMX OOB: replace ``#expanded-space`` with the Lab view."""
    return Div(
        _expanded_space(session, ICON4_ICON_ID),
        cls="flex flex-col flex-1 min-h-0 overflow-hidden w-full items-stretch justify-start",
        id="expanded-space",
        hx_swap_oob="true",
    )

## Routes

``GET /`` shows the main layout with Lab visible by default. ``GET /expand/{id}`` re-renders ``#expanded-space`` with the correct panel visible (both stay mounted; Jupyter iframe is preserved). ``POST /chat`` forwards to Lisette, appends cells via Jupyter Server when possible, and returns status (plus OOB updates mirroring ``string therapy1``).## Routes

``GET /`` shows the main layout with Lab visible by default. ``GET /expand/{id}`` re-renders ``#expanded-space`` with the correct panel visible (both stay mounted; Jupyter iframe is preserved). ``POST /chat`` forwards to Lisette, appends cells via Jupyter Server when possible, and returns status (plus OOB updates mirroring ``string therapy1``).

In [ ]:
#| export
@app.get("/")
def home(session):
    nb_default = _chat_notebook_default(session)
    _exp_cls = (
        "flex flex-col flex-1 min-h-0 overflow-hidden w-full "
        "items-stretch justify-start"
    )
    expanded_main = _expanded_space(session, ICON4_ICON_ID)
    return Title("BV"), Container(
        Div(
            cls="flex flex-col h-[100dvh] max-h-[100dvh] overflow-hidden box-border gap-1 px-2 py-2",
        )(
            Div(
                Card(
                    Div(
                        expanded_main,
                        cls=_exp_cls,
                        id="expanded-space",
                    ),
                    cls="w-full flex-1 min-h-0 flex flex-col overflow-hidden bg-base-200/50",
                    body_cls="flex flex-col flex-1 min-h-0 overflow-hidden p-1",
                ),
                cls="relative w-full flex-1 min-h-0 flex flex-col",
            ),
            _app_icon_strip(),
            Div(
                Form(
                    Input(name="vault_context", id="vault-chat-context", value="", type="hidden"),
                    Input(name="vault_group_mode", id="vault-chat-group-mode", value="0", type="hidden"),
                    hx_post="/chat",
                    hx_target="#chat-jupyter-status",
                    hx_swap="innerHTML",
                    hx_push_url="false",
                    hx_include=(
                        "#active-icon-input, #vault-chat-context, #vault-chat-group-mode, "
                        "#chat-notebook-path, #chat-jupyter-cell-type"
                    ),
                    cls="flex h-[9.5rem] max-h-[32vh] shrink-0 items-stretch gap-2 rounded-lg border border-[#3f3f3f] bg-[#222] p-2",
                )(
                    Textarea(
                        name="msg",
                        placeholder="Type your note or code here...",
                        cls="textarea min-h-0 flex-1 resize-none border-[#555] bg-[#333] text-white text-xs placeholder:text-white/60 leading-snug",
                        rows=2,
                    ),
                    Div(cls="flex w-[10.5rem] shrink-0 flex-col justify-between gap-1")(
                        Div(cls="flex flex-col gap-1")(
                            Span(
                                "Notebook path",
                                cls="text-[9px] font-medium uppercase tracking-wide text-white/50",
                            ),
                            Input(
                                name="notebook",
                                type="text",
                                value=nb_default,
                                id="chat-notebook-path",
                                placeholder="notebook.ipynb",
                                autocomplete="off",
                                cls="input input-sm w-full border-[#555] bg-[#333] text-white text-xs placeholder:text-white/40 h-7 py-0",
                                onkeydown="if(event.key==='Enter'){event.preventDefault();this.blur();}",
                            ),
                            Span(
                                "Reply as cell",
                                cls="text-[9px] font-medium uppercase tracking-wide text-white/50",
                            ),
                            Select(
                                name="jupyter_cell_type",
                                id="chat-jupyter-cell-type",
                                cls="select select-sm w-full border-[#555] bg-[#333] text-white text-xs min-h-0 h-7 py-0",
                            )(
                                Option("Markdown", value="markdown"),
                                Option("Code", value="code"),
                            ),
                            P(
                                "Chat uses the Lab assistant; vault context fields are reserved for future graph integration.",
                                cls="text-[10px] text-white/50 leading-tight",
                            ),
                        ),
                        Div(cls="flex flex-col gap-1")(
                            Button("Send", type="submit", cls="btn btn-primary btn-xs w-full font-bold h-7 min-h-0"),
                            Span(
                                "",
                                id="chat-jupyter-status",
                                cls="text-center text-[10px] min-h-[0.875rem] block text-white/40 leading-tight",
                            ),
                        ),
                    ),
                ),
                cls="shrink-0 rounded-lg border border-[#3f3f3f] bg-[#1a1a1a] p-2 shadow-lg",
            ),
        ),
        cls="max-w-full p-0",
    )


@rt("/expand/{id:int}")
def get_expand(id: int, session):
    session["active_icon"] = int(id)
    return _expanded_space(session, int(id))


@rt("/chat")
async def post_chat(
    msg: str,
    session,
    active_icon: str = None,
    vault_context: str = "",
    notebook: str | None = None,
    jupyter_cell_type: str = "markdown",
):
    if not msg.strip():
        return _chat_status_inner_html("", ok=None)

    dom_icon_id = None
    if (active_icon or "").strip():
        try:
            dom_icon_id = int(str(active_icon).strip())
            session["active_icon"] = dom_icon_id
        except (ValueError, TypeError):
            pass

    max_tokens = int(os.getenv("LISETTE_MAX_TOKENS", "2000"))
    vc = (vault_context or "").strip()
    llm_msg = msg.strip()
    if vc:
        llm_msg = (
            "The user attached excerpts from vault notes (markdown):\n\n"
            f"{vc}\n\n---\n\nUser message:\n{llm_msg}"
        )
    assistant_text, oob_update = chat_icon4(session, llm_msg, max_tokens=max_tokens)

    nb_path = resolve_notebook_path(notebook, session)
    session[SESSION_NOTEBOOK_KEY] = nb_path

    inject_ok, inject_msg = await append_chat_turn_to_notebook(
        nb_path,
        msg.strip(),
        jupyter_cell_type,
        assistant_text if assistant_text else None,
    )

    main = _chat_status_inner_html(inject_msg, ok=inject_ok)
    if inject_ok:
        if dom_icon_id == ICON4_ICON_ID:
            return main
        return main, _expanded_space_icon4_oob(session)
    if oob_update and dom_icon_id == ICON4_ICON_ID:
        return main, oob_update
    return main


def _prioritize_routes_before_static() -> None:
    """FastHTML's catch-all ``/{fname:path}.{ext}`` is registered first; it steals ``/favicon.ico`` and returns 404 if the file is missing."""
    routes = list(app.router.routes)
    prio_paths = ("/favicon.ico", "/health")
    prio = [r for r in routes if getattr(r, "path", None) in prio_paths]
    rest = [r for r in routes if getattr(r, "path", None) not in prio_paths]
    prio.sort(key=lambda r: prio_paths.index(getattr(r, "path", "")))
    app.router.routes = prio + rest


_prioritize_routes_before_static()

## ``main()``

Starts **uvicorn** on ``ST_BIND_HOST:ST_PORT`` (defaults ``0.0.0.0:5007``). Run ``jupyter lab`` yourself; icon 4 shows a link (``ST_JUPYTER_LAB_URL`` / placeholder). Use ``http://127.0.0.1:ST_PORT/`` or your host IP when not browsing on the same machine.## ``main()``

Starts **uvicorn** on ``ST_BIND_HOST:ST_PORT`` (defaults ``0.0.0.0:5007``). Run ``jupyter lab`` yourself; icon 4 shows a link (``ST_JUPYTER_LAB_URL`` / placeholder). Use ``http://127.0.0.1:ST_PORT/`` or your host IP when not browsing on the same machine.

In [ ]:
#| export
def main() -> None:
    """Start the web server. Run: ``python -m string_therapy.app``."""
    url = tc.web_app_url()
    if tc.open_browser():
        threading.Timer(1.0, lambda u=url: webbrowser.open(u)).start()
    serve(
        appname="string_therapy.app",
        host=tc.http_bind_host(),
        port=tc.http_port(),
        reload=tc.uvicorn_reload(),
        log_level="info",
    )

In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    main()